In [ ]:
#| hide
using MIM_GA

# utilities

Export step: write a `Bathymetry` raster together with zero or more
vector feature layers (e.g. the `DataFrame`s returned by
`morphological_high`/`morphological_low`/`polygonize_filtered`) into a
single GeoPackage (`.gpkg`) file, so a whole analysis session's raster
and vector results can be opened together in GIS software (QGIS,
ArcGIS) instead of as separate files with no enforced correspondence.

In [ ]:
#| default_exp utilities

In [ ]:
#| export
using Rasters
import ArchGDAL
using DataFrames

In [ ]:
#| exporti
# Map a DataFrame column's element type to the OGR field type used for the
# matching GeoPackage attribute column. Falls through to OFTString for
# anything that isn't an Integer/AbstractFloat/AbstractString subtype,
# since GDAL can always store a string and silent data loss is worse than
# a wide text column.
function _oft_for(::Type{T}) where {T}
    if T <: AbstractString
        return ArchGDAL.OFTString
    elseif T <: Integer
        return ArchGDAL.OFTInteger64
    elseif T <: AbstractFloat
        return ArchGDAL.OFTReal
    else
        return ArchGDAL.OFTString
    end
end

In [ ]:
#| exporti
# Create one GPKG vector table ("layer" in GDAL/OGR terms) from a
# DataFrame with a :geometry column, and write every row into it. Every
# other column becomes an attribute field, typed via `_oft_for` -- so
# :area, :kind, or any later Characterise-step column (GA-SaMMT Tables
# 2-4) comes along automatically with no per-column wiring needed here.
# `srs` is passed in rather than re-derived per layer so every layer in
# one export shares exactly the raster's own CRS.
function _write_vector_layer!(dataset, name::AbstractString, df::DataFrame, srs)
    hasproperty(df, :geometry) || throw(ArgumentError(
        "layer \"$name\" DataFrame has no :geometry column"))
    attr_names = [n for n in names(df) if n != "geometry"]

    # An empty DataFrame still needs a geometry type to create the layer
    # with; Polygon matches every Map-step output this package produces.
    geomtype = isempty(df.geometry) ? ArchGDAL.wkbPolygon : ArchGDAL.getgeomtype(df.geometry[1])
    layer = ArchGDAL.createlayer(name=name, dataset=dataset, geom=geomtype, spatialref=srs)
    for attr in attr_names
        ArchGDAL.addfielddefn!(layer, attr, _oft_for(eltype(df[!, attr])))
    end
    for row in eachrow(df)
        ArchGDAL.addfeature(layer) do feature
            ArchGDAL.setgeom!(feature, row.geometry)
            for (i, attr) in enumerate(attr_names)
                val = row[attr]
                ArchGDAL.setfield!(feature, i - 1, val isa AbstractString ? String(val) : val)
            end
        end
    end
    return layer
end

In [ ]:
#| export
"""
    export_geopackage(path, bathy::Bathymetry; layers=Dict(), raster_name="bathymetry") -> path

Write `bathy`'s raster together with zero or more vector feature layers
into a single GeoPackage file at `path`. `layers` maps a GPKG table name
to a `DataFrame` with a `:geometry` column -- exactly what
`morphological_high`/`morphological_low`/`polygonize_filtered` return,
including a `vcat` of several such frames (e.g. tagged `:kind` for
"high"/"low" before concatenating). Every non-`:geometry` column becomes
an attribute field.

GeoPackage's raster ("tiles") format only supports Byte/Int16/UInt16/
Float32 bands, so `bathy`'s data is narrowed to Float32 before writing --
standard practice for depth data, and far below any sonar/LIDAR noise
floor. The raster is round-tripped through a `/vsimem/` GeoTIFF first
(the same pattern `polygonize_filtered` uses) rather than writing GPKG
directly: GDAL's GPKG raster driver validates the geotransform only at
dataset close and errors on anything but a standard north-up
orientation, while `Rasters.write` to GeoTIFF is unconditionally
reliable regardless of axis order, and `ArchGDAL.copy` (`GDALCreateCopy`)
from that GeoTIFF into the GPKG driver is what actually produces a valid
GPKG raster table. Verified against a real south-up synthetic raster
(errors, confirming the GPKG driver's own restriction, not a bug here)
and a north-up one (succeeds) -- real bathymetry loaded through
`SeafloorMap`/`Bathymetry` from a GeoTIFF is north-up already, so this
is not expected to surface in practice.

Every vector layer is written in the raster's own CRS (read back off the
GDAL dataset that was just created, not re-derived from `bathy`
independently, so the two can never disagree). `layers` can be empty, in
which case only the raster is written.

Overwrites `path` if it already exists.
"""
function export_geopackage(path::AbstractString, bathy::Bathymetry;
                            layers::AbstractDict{<:AbstractString}=Dict{String,DataFrame}(),
                            raster_name::AbstractString="bathymetry")
    isfile(path) && rm(path)

    clean = replace_missing(bathy.raster, NaN32)
    raster32 = Rasters.rebuild(clean; data=Float32.(Array(clean)), missingval=NaN32)

    vsimem_path = "/vsimem/mim_ga_export_$(time_ns())_$(rand(UInt32)).tif"
    crs_wkt = ""
    try
        Rasters.write(vsimem_path, raster32; force=true)
        ArchGDAL.read(vsimem_path) do src_ds
            crs_wkt = ArchGDAL.getproj(src_ds)
            ArchGDAL.copy(src_ds; filename=path, driver=ArchGDAL.getdriver("GPKG"),
                          options=["RASTER_TABLE=$raster_name"]) do dst_ds
            end
        end
    finally
        ArchGDAL.GDAL.vsiunlink(vsimem_path)
    end

    isempty(layers) && return path

    srs = ArchGDAL.importWKT(crs_wkt)
    ArchGDAL.update(path) do dataset
        for (name, df) in layers
            _write_vector_layer!(dataset, name, df, srs)
        end
    end
    return path
end